# Supplementary Figure 14 - TNBC subgroup heatmaps

            This notebook is a cleaned, English-commented manuscript code file.

            **Provenance.** `Ali_TNBC_2023/NTPublic/sfplot/sfplot_TNBC.ipynb` and local/A100 mirrors.

            The notebook keeps the original manuscript data paths where they were found. If a path points to
            `/data/taobo.hu` or `/mnt/taobo.hu`, run the notebook on the A100 server or adapt the path to the
            matching mounted Y-drive location.


In [ ]:
library(data.table)
library(dplyr)
library(tidyr)
library(ggplot2)

PROJECT_DIR <- "Y:/long/publication_datasets/Ali_TNBC_2023/NTPublic"
setwd(PROJECT_DIR)
source(here::here("code/header.R"))
source("Y:/long/publication_datasets/Vannan_2023_Lung_Fibrosis/Rcode/sfplotR/R/compute_cophenetic_distances_from_df.R")
source("Y:/long/publication_datasets/Vannan_2023_Lung_Fibrosis/Rcode/sfplotR/R/plot_cophenetic_heatmap.R")

cells <- as.data.frame(getCells(curated = TRUE, allCells = TRUE))
clinical <- read.csv(file.path(PROJECT_DIR, "data/derived/clinical.csv"))


In [ ]:
OUTPUT_DIR <- "outputs/supp_fig_14_TNBC_subgroup_heatmaps"
dir.create(file.path(OUTPUT_DIR, "Searcher"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(OUTPUT_DIR, "Findee"), recursive = TRUE, showWarnings = FALSE)

unique_groups <- unique(clinical[, c("pCR", "Arm")])
for (group_idx in seq_len(nrow(unique_groups))) {
  current_pCR <- as.character(unique_groups$pCR[group_idx])
  current_Arm <- as.character(unique_groups$Arm[group_idx])
  patient_ids <- unique(clinical$PatientID[clinical$pCR == current_pCR & clinical$Arm == current_Arm])
  for (phase in unique(cells$BiopsyPhase)) {
    row_matrices <- list(); col_matrices <- list()
    for (patient in patient_ids) {
      submeta <- subset(cells, PatientID == patient & BiopsyPhase == phase)
      if (nrow(submeta) < 50) next
      result <- compute_cophenetic_distances_from_df(submeta, cluster_col = "Label", x_col = "Location_Center_X", y_col = "Location_Center_Y", method = "average")
      row_matrices[[patient]] <- result$row_cophenetic_df
      col_matrices[[patient]] <- result$col_cophenetic_df
    }
    if (length(row_matrices) == 0) next
    avg_row <- Reduce(`+`, row_matrices) / length(row_matrices)
    avg_col <- Reduce(`+`, col_matrices) / length(col_matrices)
    suffix <- paste(phase, current_pCR, paste0("Arm_", current_Arm), sep = "_")
    plot_cophenetic_heatmap(avg_row, matrix_name = "row_coph", output_dir = file.path(OUTPUT_DIR, "Searcher"), sample = suffix, parse_label = TRUE)
    plot_cophenetic_heatmap(avg_col, matrix_name = "col_coph", output_dir = file.path(OUTPUT_DIR, "Findee"), sample = suffix, parse_label = TRUE)
  }
}
